<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2006%20-%202026-04-28%20-%20Pandas%20I/clase_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 6 · Pandas I

Hasta ayer trabajaste con **objetos** y con listas/diccionarios de Python. Hoy das el salto a **Pandas**, la librería que vas a vivir el resto del curso. Si Excel fuera código, sería esto — pero más honesto, más rápido y reproducible.

Pandas se construye sobre dos estructuras: **Series** (un vector etiquetado, 1D) y **DataFrame** (una tabla con filas y columnas, 2D). Hoy vas a aprender a crearlas, cargarlas desde archivos, explorarlas, seleccionar partes y filtrarlas con condiciones. Ese es el 80% de lo que hace cualquier analista de datos en su día a día.

> **Hoy haces** · La teoría comentada, **cuatro ejercicios guiados** (Series, DataFrame, exploración, filtros), **tres desafíos de profundización** y un checkpoint final. ~2h30.
>
> **Entrega** · El notebook ejecutado de principio a fin, con tus respuestas en las celdas marcadas, subido al repo del equipo antes de la próxima clase.

## 📺 Video de referencia

Antes de empezar, mira los primeros 25 minutos de este video. Cubre exactamente los conceptos de hoy con ejemplos visuales:

**[Curso de Pandas en Python — píldoras informáticas](https://www.youtube.com/watch?v=fW4olN1JWzQ)**

[![Pandas Python en Español](https://img.youtube.com/vi/fW4olN1JWzQ/0.jpg)](https://www.youtube.com/watch?v=fW4olN1JWzQ)

> *Video en español, ritmo pausado y muchos ejemplos. Si prefieres lectura, en `lecturas/` tienes el tutorial completo de Jose R. Zapata como respaldo.*

In [ ]:
# --- Setup del entorno ---
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)

print("Setup completo ✓")
print(f"pandas {pd.__version__} · numpy {np.__version__}")

---

## 1. ¿Qué es Pandas y por qué lo necesitas?

Hasta ahora trabajaste con **listas**, **diccionarios** y **clases**. Eso funciona para datos pequeños, pero cuando tienes 50 mil filas con tipos mixtos y valores faltantes, las listas Python se quedan cortas: son lentas, no entienden de etiquetas, y no traen nada listo para limpiar, agrupar o cruzar datos.

Pandas resuelve eso. Es una librería de **manipulación de datos tabulares** construida sobre NumPy y, ahora, sobre Apache Arrow. Te da dos estructuras clave y un montón de métodos optimizados encima.

| Estructura | Dimensiones | Analogía | Equivalente en Excel |
|---|---|---|---|
| **`Series`** | 1D (vector etiquetado) | Una columna con índice | Una columna |
| **`DataFrame`** | 2D (tabla) | Diccionario de Series con el mismo índice | Una hoja entera |

```
   Series                  DataFrame
   ──────                  ─────────
   índice  valor           índice  col_A  col_B  col_C
   0       12              0       12     'a'    9.1
   1        7              1        7     'b'    3.4
   2       19              2       19     'c'    7.8
   3        4              3        4     'd'    2.0
```

**Lo que vas a poder hacer con Pandas (y no con listas):**

- Leer y escribir CSV, JSON, Excel, Parquet o SQL en una línea.
- Seleccionar filas y columnas por **etiqueta** (no solo por posición).
- Filtrar con condiciones booleanas vectorizadas.
- Manejar valores faltantes (`NaN`) sin escribir loops.
- Agrupar, pivotar y unir datasets como en SQL.
- Vectorizar operaciones — el mismo cálculo aplica a millones de filas en milisegundos.

---

## 2. `Series`: el ladrillo de Pandas

Una **Series** es un vector con etiquetas. Puedes pensarlo como una mezcla entre una lista y un diccionario: tiene **valores** (como una lista) y un **índice** (como las claves de un diccionario).

In [ ]:
# Tres formas distintas de crear la misma Series

# 2.1 Desde una lista (índice automático 0,1,2…)
s_lista = pd.Series([10, 20, 30, 40])
print("Desde lista:")
print(s_lista)
print()

# 2.2 Desde una lista con índice personalizado
poblacion = pd.Series(
    data=[2_780_000, 1_650_000, 580_000, 380_000],
    index=["Quito", "Guayaquil", "Cuenca", "Loja"],
    name="poblacion_2024"
)
print("Con índice de ciudades:")
print(poblacion)
print()

# 2.3 Desde un diccionario (las llaves se convierten en el índice)
ventas = pd.Series({"ene": 1200, "feb": 1500, "mar": 980, "abr": 1750})
print("Desde dict:")
print(ventas)

### Acceder a los datos: por etiqueta o por posición

Como una Series es lista *y* diccionario al mismo tiempo, puedes acceder de las dos formas. Lo importante: las **operaciones se alinean por índice**, no por posición — eso es lo que la diferencia de un array NumPy.

In [ ]:
# Acceso por etiqueta (como diccionario)
print(f"Quito tiene {poblacion['Quito']:,} habitantes")

# Selección múltiple
print()
print("Tres ciudades grandes:")
print(poblacion[["Quito", "Guayaquil", "Cuenca"]])

# Operaciones vectorizadas — se aplican a TODOS los valores a la vez
print()
print("Población en miles:")
print((poblacion / 1000).round(0))

# Estadísticas integradas
print()
print(f"Total: {poblacion.sum():,}")
print(f"Promedio: {poblacion.mean():,.0f}")
print(f"Máximo: {poblacion.max():,} → {poblacion.idxmax()}")

### Alineación por índice

Cuando sumas dos Series, Pandas **empareja por etiqueta**, no por orden de aparición. Si una etiqueta no existe en ambas, el resultado es `NaN` (Not a Number — el marcador de "valor faltante" en Pandas).

In [ ]:
ventas_2023 = pd.Series({"Quito": 1200, "Guayaquil": 1500, "Cuenca": 600})
ventas_2024 = pd.Series({"Quito": 1500, "Guayaquil": 1800, "Loja": 400})

print("Suma alineada por etiqueta (Cuenca y Loja sólo están en una):")
print(ventas_2023 + ventas_2024)

---

## 3. `DataFrame`: tablas con superpoderes

Un **DataFrame** es un conjunto de Series que comparten el mismo índice. Es decir: una tabla con filas, columnas y un índice por fila. Cada columna puede tener un **tipo distinto** (numérica, texto, fecha, booleano…).

Hay muchas formas de construir un DataFrame. La más común en la vida real es **leer un archivo**; pero también puedes crearlos desde diccionarios, listas o arrays.

In [ ]:
# 3.1 Desde un diccionario de listas (cada clave es una columna)
empleados = pd.DataFrame({
    "nombre":  ["Ana", "Luis", "Carla", "Pedro", "Marta"],
    "ciudad":  ["Quito", "Guayaquil", "Quito", "Quito", "Cuenca"],
    "area":    ["Tech", "Tech", "Ventas", "Tech", "Ventas"],
    "salario": [1800, 2100, 1500, 2400, 1700],
    "edad":    [28, 34, 25, 41, 30],
})

empleados

In [ ]:
# 3.2 Desde una lista de diccionarios (cada dict es una FILA)
ordenes = pd.DataFrame([
    {"orden_id": 1001, "producto": "Laptop",    "unidades": 1, "precio": 899.0},
    {"orden_id": 1002, "producto": "Mouse",     "unidades": 3, "precio":  29.5},
    {"orden_id": 1003, "producto": "Teclado",   "unidades": 2, "precio":  79.9},
    {"orden_id": 1004, "producto": "Monitor",   "unidades": 1, "precio": 349.0},
])

ordenes

**Anatomía de un DataFrame:**

```
        ┌──────────────── columnas (Series) ───────────────┐
        │                                                   │
        nombre   ciudad     area    salario  edad   ← .columns
índice ─┼──────  ─────────  ──────  ───────  ────
   0    Ana      Quito      Tech     1800     28
   1    Luis     Guayaquil  Tech     2100     34   ← una FILA
   2    Carla    Quito      Ventas   1500     25
   3    Pedro    Quito      Tech     2400     41
   4    Marta    Cuenca     Ventas   1700     30
   │
  .index
```

Cada columna es una `Series` — y de hecho, si pides una columna, eso es lo que te devuelve.

In [ ]:
col_salario = empleados["salario"]
print(type(col_salario))
print()
print(col_salario)

---

## 4. Cargar datos reales

En la práctica casi nunca creas DataFrames a mano. Los **lees** de un archivo. Pandas trae lectores para los formatos más comunes:

| Función | Para qué sirve |
|---|---|
| `pd.read_csv("archivo.csv")` | CSV (separador coma) |
| `pd.read_excel("archivo.xlsx")` | Excel (requiere `openpyxl`) |
| `pd.read_json("archivo.json")` | JSON |
| `pd.read_parquet("archivo.parquet")` | Parquet (columnar, comprimido) |
| `pd.read_sql(query, conn)` | Bases SQL |

Para escribir, todos tienen su versión `to_*`: `df.to_csv(...)`, `df.to_excel(...)`, etc.

Para hoy vamos a usar el dataset **Titanic** de Seaborn — ya está incluido y no necesitas descargar nada. 891 pasajeros, 15 columnas, valores faltantes incluidos. El escenario clásico de EDA.

In [ ]:
# Cargamos el dataset desde seaborn (equivale a un read_csv)
titanic = sns.load_dataset("titanic")

# Equivalente con read_csv (lo dejamos comentado como referencia):
# titanic = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")

# Echamos un primer vistazo
titanic.head()

---

## 5. Exploración rápida: los 5 métodos que SIEMPRE corres primero

Cuando llega un dataset nuevo, **antes de nada** corres estos cinco métodos. En 30 segundos sabes contra qué te enfrentas: cuántas filas hay, qué tipos hay, qué falta y qué rango toman los valores.

In [ ]:
# 1. Dimensiones (filas, columnas)
print(f".shape       → {titanic.shape}  (filas, columnas)")
print(f".size        → {titanic.size}   (celdas totales)")
print(f".columns     → {list(titanic.columns)}")
print(f".index       → {titanic.index}")
print(f".dtypes      ↓")
print(titanic.dtypes)

In [ ]:
# 2. Información general (tipos + nulos en una sola vista)
titanic.info()

In [ ]:
# 3. Estadísticas descriptivas de columnas numéricas
titanic.describe().round(2)

In [ ]:
# 4. Estadísticas para columnas categóricas (texto)
titanic.describe(include="object").T

In [ ]:
# 5. Conteo de nulos por columna (¡el primer diagnóstico!)
nulos = titanic.isnull().sum()
nulos_pct = (nulos / len(titanic) * 100).round(1)

resumen_nulos = pd.DataFrame({"nulos": nulos, "porcentaje": nulos_pct})
resumen_nulos = resumen_nulos[resumen_nulos["nulos"] > 0].sort_values("nulos", ascending=False)
resumen_nulos

**Lectura rápida del Titanic:**

- 891 filas × 15 columnas — manejable en memoria.
- `age` tiene **177 nulos** (~20%): habrá que decidir qué hacer con ellos.
- `deck` tiene **688 nulos** (~77%): tan rota que normalmente se descarta.
- `embarked` y `embark_town` tienen sólo 2 nulos: los podemos imputar o eliminar sin drama.
- La tarifa promedio (`fare`) es 32, pero el máximo es 512 — hay outliers fuertes.

Nada de esto requirió cargar ni una sola línea de pandas avanzado. Sólo `info()`, `describe()` y un poco de `isnull().sum()`.

In [ ]:
# Vistazos adicionales útiles
print("Primeras 3 filas:")
print(titanic.head(3))
print()
print("Últimas 3 filas:")
print(titanic.tail(3))
print()
print("3 filas aleatorias (útil cuando los datos están ordenados):")
print(titanic.sample(3, random_state=SEED))

---

## 6. Seleccionar columnas y crear nuevas

Una vez que entiendes la forma del DataFrame, lo siguiente es **acceder a partes** y **derivar columnas nuevas**. Pandas vectoriza todas las operaciones, así que no necesitas loops.

In [ ]:
# Una columna → devuelve una Series
edades = titanic["age"]
print(type(edades), edades.head().tolist())

# Varias columnas → devuelve un DataFrame (¡fíjate en los corchetes dobles!)
subset = titanic[["sex", "age", "fare", "survived"]]
subset.head()

In [ ]:
# Crear columnas nuevas a partir de otras (operación vectorizada)
empleados["salario_anual"] = empleados["salario"] * 12
empleados["es_tech"] = empleados["area"] == "Tech"
empleados["categoria_salarial"] = pd.cut(
    empleados["salario"],
    bins=[0, 1600, 2000, 10_000],
    labels=["junior", "mid", "senior"]
)

empleados

### Eliminar columnas

Tres formas, todas válidas:

```python
df.drop("col", axis="columns")          # devuelve una COPIA sin esa columna
df.drop("col", axis="columns", inplace=True)  # modifica el DataFrame original
del df["col"]                            # in-place; menos común
```

Recomendación: usar `df = df.drop(...)` (sin `inplace=True`). Es más explícito y se encadena mejor.

In [ ]:
empleados_sin_anual = empleados.drop(columns=["salario_anual", "categoria_salarial"])
empleados_sin_anual.head()

---

## 7. `loc` vs `iloc`: el clásico que confunde a todos

Pandas tiene **dos** indexadores principales para seleccionar filas y columnas. Son distintos y vale la pena memorizar la diferencia desde hoy.

| Indexador | Selecciona por | Ejemplo |
|---|---|---|
| **`.loc[fila, columna]`** | **etiqueta** (nombre del índice) | `df.loc[3, "age"]` |
| **`.iloc[fila, columna]`** | **posición** (entero, como una lista) | `df.iloc[3, 5]` |

Regla mnemotécnica:

- **`loc`** = **l**abel = etiqueta
- **`iloc`** = **i**nteger location = posición numérica

```
            loc["A", "X"]               iloc[0, 0]
           ↓ etiqueta etiqueta         ↓ pos pos
       X    Y    Z                    X   Y   Z
   A   1.2  3.4  5.6                  1.2 3.4 5.6
   B   2.1  4.2  6.3                  2.1 4.2 6.3
   C   3.0  5.1  7.0                  3.0 5.1 7.0
```

In [ ]:
# Vamos a poner un índice más significativo para ver la diferencia
demo = empleados.set_index("nombre")[["ciudad", "area", "salario", "edad"]]
demo

In [ ]:
# .loc — por ETIQUETA del índice y nombre de columna
print(".loc['Pedro', 'salario'] →", demo.loc["Pedro", "salario"])
print()
print("Fila completa de Carla:")
print(demo.loc["Carla"])
print()
print("Slice por etiquetas (¡los dos extremos están INCLUIDOS!):")
print(demo.loc["Ana":"Carla", ["ciudad", "salario"]])

In [ ]:
# .iloc — por POSICIÓN (igual que listas/arrays)
print(".iloc[3, 2] →", demo.iloc[3, 2])  # fila 3, columna 2
print()
print("Primeras 3 filas, primeras 2 columnas:")
print(demo.iloc[:3, :2])
print()
print("Última fila:")
print(demo.iloc[-1])

**Tres reglas de oro:**

1. Si tu índice tiene nombres con sentido (fechas, IDs, ciudades) → usa `loc`. Es más legible.
2. Si quieres la fila *N* o la columna *N* sin importar la etiqueta → usa `iloc`.
3. **Nunca mezcles**: `df.loc[3, "col"]` funciona si el índice incluye `3` como etiqueta, pero confunde mucho. Para enteros como posición, usa siempre `iloc`.

---

## 8. Filtros booleanos: la herramienta más usada del día a día

Para responder a "dame todas las filas donde X cumple Y", Pandas usa **indexación booleana**: pasas una Series de `True/False` del mismo largo que el DataFrame y obtienes solo las filas marcadas con `True`.

In [ ]:
# Paso 1: construir la condición → devuelve una Series booleana
condicion = titanic["age"] > 60
print(condicion.head(10))
print(f"\nTotal de pasajeros mayores de 60: {condicion.sum()}")

In [ ]:
# Paso 2: usar esa condición como filtro
mayores_60 = titanic[titanic["age"] > 60]
print(f"{len(mayores_60)} filas seleccionadas")
mayores_60[["sex", "age", "pclass", "survived"]].head()

### Combinar varias condiciones

Cuando hay más de una condición, usa los operadores **vectorizados** y **siempre** pon paréntesis alrededor de cada parte:

| Operador booleano | Pandas | Equivalente Python |
|---|---|---|
| Y lógico | `&` | `and` |
| O lógico | `\|` | `or` |
| Negación | `~` | `not` |

> Si olvidas los paréntesis, el error te va a confundir. Es el bug más común al empezar con Pandas.

In [ ]:
# Mujeres de primera clase que sobrevivieron
mujeres_1ra_vivas = titanic[
    (titanic["sex"] == "female") &
    (titanic["pclass"] == 1) &
    (titanic["survived"] == 1)
]
print(f"Mujeres en clase 1 que sobrevivieron: {len(mujeres_1ra_vivas)}")
print(f"Edad promedio: {mujeres_1ra_vivas['age'].mean():.1f} años")
mujeres_1ra_vivas.head(3)

In [ ]:
# Negación con ~ (todo lo que NO es de Cherbourg)
sin_cherbourg = titanic[~(titanic["embark_town"] == "Cherbourg")]
print(f"Pasajeros que NO embarcaron en Cherbourg: {len(sin_cherbourg)}")

# Pertenencia con .isin()
puertos_grandes = titanic[titanic["embark_town"].isin(["Southampton", "Cherbourg"])]
print(f"Embarcados en Southampton o Cherbourg: {len(puertos_grandes)}")

### `.query()`: filtros legibles cuando las condiciones se complican

Cuando las condiciones empiezan a anidarse, `.query()` te deja escribir el filtro como un string. Más corto, más legible — sobre todo si lo van a leer otras personas.

In [ ]:
# Las dos formas son EQUIVALENTES
clasico = titanic[(titanic["age"] > 30) & (titanic["fare"] < 50) & (titanic["sex"] == "male")]
con_query = titanic.query("age > 30 and fare < 50 and sex == 'male'")

print(f"Clásico: {len(clasico)} filas")
print(f"Query  : {len(con_query)} filas")
print(f"¿Iguales? {clasico.equals(con_query)}")

---

## 9. Cambiar el índice: `set_index` y `reset_index`

A veces tu DataFrame tiene una columna que **funciona mejor como índice** (un `id`, una fecha, un código de país). Otras veces el índice se complicó después de filtrar y quieres reiniciarlo. Para eso están estos dos métodos.

In [ ]:
# Convertir 'nombre' en el índice
emp_por_nombre = empleados.set_index("nombre")
emp_por_nombre.head(3)

In [ ]:
# Reiniciar el índice → vuelve a 0,1,2,…
emp_reiniciado = emp_por_nombre.reset_index()
emp_reiniciado.head(3)

---

## 🌶️ Ejercicio guiado 1: Series con datos económicos

Construye una `Series` con la inflación anual de tres países y respóndele a tu jefe tres preguntas básicas. Reemplaza los `___` por la sintaxis correcta.

| País | Inflación 2024 (%) |
|---|---|
| Ecuador | 2.4 |
| Colombia | 7.1 |
| Perú | 3.8 |
| Chile | 3.2 |

In [ ]:
# 1. Crea la Series con los nombres de país como índice
inflacion = pd.Series(
    data=___,
    index=___,
    name="inflacion_2024"
)

# 2. ¿Cuál es la inflación de Colombia?  (acceso por etiqueta)
inflacion_colombia = ___

# 3. ¿Cuál es el país con MAYOR inflación?
pais_max = inflacion.___()

# 4. Convierte a tasa decimal (divide cada valor entre 100)
inflacion_decimal = ___

print("Series:")
print(inflacion)
print(f"\nColombia: {inflacion_colombia}%")
print(f"Máximo: {pais_max}")
print(f"\nDecimal:\n{inflacion_decimal}")

In [ ]:
# Validaciones
assert isinstance(inflacion, pd.Series), "`inflacion` debe ser una Series"
assert list(inflacion.index) == ["Ecuador", "Colombia", "Perú", "Chile"], "Índice incorrecto"
assert inflacion_colombia == 7.1, "Inflación de Colombia incorrecta"
assert pais_max == "Colombia", "País con mayor inflación incorrecto"
assert (inflacion_decimal.round(4) == (inflacion / 100).round(4)).all(), "La conversión decimal está mal"
print("✓ Ejercicio 1 completado")

<details><summary>💡 Solución sugerida (ábrela solo si te atascas)</summary>

```python
inflacion = pd.Series(
    data=[2.4, 7.1, 3.8, 3.2],
    index=["Ecuador", "Colombia", "Perú", "Chile"],
    name="inflacion_2024"
)
inflacion_colombia = inflacion["Colombia"]
pais_max = inflacion.idxmax()
inflacion_decimal = inflacion / 100
```

</details>

---

## 🌶️ Ejercicio guiado 2: Construir un DataFrame de ventas

Te entregan los datos de ventas de cinco tiendas. Quieres armar un DataFrame y responder algunas preguntas. Sigue los pasos.

In [ ]:
# Datos crudos
datos_tiendas = {
    "tienda":   ["T001", "T002", "T003", "T004", "T005"],
    "ciudad":   ["Quito", "Guayaquil", "Cuenca", "Quito", "Loja"],
    "ventas":   [12_500, 18_200, 7_900, 14_300, 6_400],
    "empleados": [8, 12, 5, 10, 4],
}

# 1. Crea el DataFrame
ventas_df = pd.___(datos_tiendas)

# 2. Muestra solo las 3 primeras filas
primeras = ventas_df.___(3)

# 3. Crea una columna nueva: ventas_por_empleado = ventas / empleados
ventas_df["ventas_por_empleado"] = ___

# 4. Convierte la columna 'tienda' en el índice del DataFrame
ventas_df = ventas_df.___("tienda")

# 5. Muestra el resultado final
ventas_df

In [ ]:
assert isinstance(ventas_df, pd.DataFrame), "ventas_df debe ser DataFrame"
assert ventas_df.index.name == "tienda", "El índice debe ser 'tienda'"
assert "ventas_por_empleado" in ventas_df.columns, "Falta la columna derivada"
assert round(ventas_df.loc["T002", "ventas_por_empleado"], 4) == round(18200/12, 4), "Cálculo erróneo"
print("✓ Ejercicio 2 completado")

<details><summary>💡 Solución sugerida</summary>

```python
ventas_df = pd.DataFrame(datos_tiendas)
primeras = ventas_df.head(3)
ventas_df["ventas_por_empleado"] = ventas_df["ventas"] / ventas_df["empleados"]
ventas_df = ventas_df.set_index("tienda")
```

</details>

---

## 🌶️ Ejercicio guiado 3: Auditoría rápida del Titanic

Acabas de recibir el dataset y necesitas entregar una mini-auditoría en menos de 5 minutos. Completa los huecos para reportar las dimensiones, los tipos, los nulos y un resumen estadístico.

In [ ]:
# 1. Dimensiones del DataFrame
filas, columnas = titanic.___
print(f"El dataset tiene {filas} filas y {columnas} columnas")

# 2. Lista de columnas
print(f"\nColumnas: {list(titanic.___)}")

# 3. Tipos de datos por columna
tipos = titanic.___
print(f"\nTipos:\n{tipos}")

# 4. Porcentaje de nulos por columna (solo donde haya nulos)
pct_nulos = (titanic.___().sum() / len(titanic) * 100).round(1)
pct_nulos = pct_nulos[pct_nulos > 0].sort_values(ascending=False)
print(f"\nNulos (%):\n{pct_nulos}")

# 5. Resumen estadístico de las columnas numéricas
resumen = titanic.___()
print("\nResumen estadístico:")
print(resumen.round(2))

In [ ]:
assert filas == 891 and columnas == 15, "Dimensiones incorrectas"
assert pct_nulos.iloc[0] > 70, "La columna con más nulos (deck) debería tener > 70%"
assert resumen.loc["mean", "fare"] > 30, "El promedio de fare está mal"
print("✓ Ejercicio 3 completado")

<details><summary>💡 Solución sugerida</summary>

```python
filas, columnas = titanic.shape
print(f"Columnas: {list(titanic.columns)}")
tipos = titanic.dtypes
pct_nulos = (titanic.isnull().sum() / len(titanic) * 100).round(1)
resumen = titanic.describe()
```

</details>

---

## 🌶️ Ejercicio guiado 4: Filtros con condiciones múltiples

Vamos a responder cuatro preguntas reales sobre el Titanic usando filtros booleanos. Cada pregunta es un mini-ejercicio.

In [ ]:
# 1. ¿Cuántos hombres de tercera clase había?
hombres_3ra = titanic[(titanic["sex"] == "male") & (titanic["pclass"] == ___)]
n_hombres_3ra = len(hombres_3ra)

# 2. ¿Cuántos niños (age < 12) viajaban en total?
ninos = titanic[titanic["age"] < ___]
n_ninos = len(ninos)

# 3. ¿Cuántos pasajeros pagaron una tarifa entre 50 y 100?
tarifa_media = titanic[(titanic["fare"] >= 50) & (titanic["fare"] <= ___)]
n_tarifa_media = len(tarifa_media)

# 4. Reescribe el filtro #3 usando .query()
tarifa_media_query = titanic.query(___)

print(f"Hombres en 3ra clase: {n_hombres_3ra}")
print(f"Niños menores de 12: {n_ninos}")
print(f"Tarifas entre 50 y 100: {n_tarifa_media}")
print(f"Mismo resultado con query: {len(tarifa_media_query) == n_tarifa_media}")

In [ ]:
assert n_hombres_3ra == 347, f"Esperaba 347 hombres en 3ra clase, obtuve {n_hombres_3ra}"
assert n_ninos > 60 and n_ninos < 80, f"Niños menores de 12 entre 60 y 80, obtuve {n_ninos}"
assert n_tarifa_media > 100, f"Esperaba >100 tarifas entre 50 y 100, obtuve {n_tarifa_media}"
assert len(tarifa_media_query) == n_tarifa_media, "Query y filtro clásico no coinciden"
print("✓ Ejercicio 4 completado")

<details><summary>💡 Solución sugerida</summary>

```python
hombres_3ra = titanic[(titanic["sex"] == "male") & (titanic["pclass"] == 3)]
ninos = titanic[titanic["age"] < 12]
tarifa_media = titanic[(titanic["fare"] >= 50) & (titanic["fare"] <= 100)]
tarifa_media_query = titanic.query("fare >= 50 and fare <= 100")
```

</details>

---

## 🌶️🌶️ Desafío 1: Mini análisis de tarifas en Titanic

Sin skeleton esta vez. Tu tarea: producir un mini reporte de las tarifas (`fare`) del Titanic, con cinco hallazgos numéricos.

**Reglas del desafío:**

1. Calcula la **tarifa promedio**, la **mediana** y la **desviación estándar** de `fare`.
2. Identifica al pasajero que pagó **más caro**: imprime nombre, sexo, clase y tarifa.
3. Cuenta cuántos pasajeros viajaron **gratis** (`fare == 0`).
4. Compara la tarifa promedio entre pasajeros que **sobrevivieron** vs los que **no** sobrevivieron.
5. Crea un histograma de `fare` (con `plt.hist` o `sns.histplot`).

> **Hint 1**: `titanic["fare"].mean()`, `.median()`, `.std()`.
>
> **Hint 2**: Para encontrar al pasajero más caro, usa `titanic["fare"].idxmax()` y luego `titanic.loc[idx]`.
>
> **Hint 3**: Para comparar dos grupos, filtra dos veces y calcula la media en cada uno, o usa `titanic.groupby("survived")["fare"].mean()` (groupby viene mañana, pero si quieres adelantarte ya).

In [ ]:
# TU CÓDIGO AQUÍ — Desafío 1
# Sugerencia de variables a calcular:
#   tarifa_promedio, tarifa_mediana, tarifa_std
#   pasajero_mas_caro  (DataFrame de una fila o Series)
#   n_gratis
#   media_vivos, media_muertos

# Después del análisis, dibuja el histograma:
# plt.figure(figsize=(10, 4))
# sns.histplot(titanic["fare"], bins=40, kde=True)
# plt.title("Distribución de tarifas en Titanic")
# plt.xlabel("Tarifa (USD 1912)")
# plt.ylabel("Pasajeros")
# plt.show()


---

## 🌶️🌶️ Desafío 2: Filtrado con condiciones complejas

Construye dos *cohortes* (segmentos) con filtros y compara sus tasas de supervivencia.

**Cohorte A — "VIP"**: mujeres de primera o segunda clase, mayores de 18 años, con tarifa > 30.

**Cohorte B — "Vulnerable"**: hombres de tercera clase, menores de 35 años, sin acompañantes (`sibsp == 0` y `parch == 0`).

Para cada cohorte calcula: tamaño, tasa de supervivencia (% que sobrevivió), edad promedio y tarifa promedio. Compara los resultados en una tablita.

> **Hint 1**: Usa indexación booleana o `.query()`. Si vas con `.query()`, recuerda que las strings van entre comillas dobles externas y simples internas: `df.query("sex == 'female'")`.
>
> **Hint 2**: La **tasa de supervivencia** = `cohorte["survived"].mean() * 100` (porque `survived` es 0/1).
>
> **Hint 3**: Para comparar, arma un nuevo DataFrame con un dict: `pd.DataFrame({"VIP": [...], "Vulnerable": [...]}, index=["tamaño", "tasa", "edad", "tarifa"])`.

In [ ]:
# TU CÓDIGO AQUÍ — Desafío 2
cohorte_vip = ___
cohorte_vulnerable = ___

# def resumen(c):
#     return [len(c), round(c["survived"].mean()*100, 1), round(c["age"].mean(), 1), round(c["fare"].mean(), 1)]
#
# tabla = pd.DataFrame(
#     {"VIP": resumen(cohorte_vip), "Vulnerable": resumen(cohorte_vulnerable)},
#     index=["tamaño", "tasa_supervivencia_%", "edad_prom", "tarifa_prom"]
# )
# tabla


---

## 🌶️🌶️🌶️ Desafío 3: tabla cruzada de supervivencia

Sin pasos intermedios. Replica la pregunta clásica: **¿Cómo afectaron clase y sexo a la supervivencia?**

1. Construye una tabla cruzada con `pd.crosstab`: filas = `pclass`, columnas = `sex`, valores = `survived`, agregación = media.
2. Convierte el resultado a porcentaje (multiplica por 100).
3. Identifica el **grupo con mayor tasa** y el **grupo con menor tasa**.
4. Visualiza la tabla con `sns.heatmap` (annot=True, cmap="RdYlGn").

> **Hint 1**: `pd.crosstab(titanic["pclass"], titanic["sex"], values=titanic["survived"], aggfunc="mean")`
>
> **Hint 2**: Para identificar el extremo: `tabla.stack().idxmax()` devuelve la tupla `(fila, columna)`.
>
> **Hint 3**: Si tu heatmap sale con números cero a la derecha, prueba `fmt=".0f"` o `fmt=".1f"`.

In [ ]:
# TU CÓDIGO AQUÍ — Desafío 3
# tabla = pd.crosstab(...)
# tabla_pct = tabla * 100
# grupo_max = ...
# grupo_min = ...
# sns.heatmap(...)
# plt.show()


---

## 🏁 Checkpoint — ¿Qué aprendiste hoy?

Responde estas preguntas en la celda de abajo (en comentarios o como un string). Servirán para tu repaso.

1. ¿Cuál es la diferencia entre una **`Series`** y un **`DataFrame`**?
2. ¿Cuándo conviene usar `.loc` y cuándo `.iloc`?
3. ¿Cómo filtras filas con dos condiciones a la vez? Da un ejemplo con `&` y otro con `.query()`.
4. ¿Qué métodos usas para auditar un dataset nuevo en menos de 30 segundos?
5. ¿Por qué el resultado de `df["col"] > 0` es una Series booleana, y cómo se usa para filtrar?

In [ ]:
# Escribe tus respuestas aquí

# 1. Series vs DataFrame:

# 2. loc vs iloc:

# 3. Filtros con & y query():

# 4. Auditoría rápida:

# 5. Series booleana e indexación:


In [ ]:
# Validación final del notebook
assert titanic.shape == (891, 15), "El dataset Titanic se modificó"
assert isinstance(empleados, pd.DataFrame), "Falta el DataFrame empleados"
assert isinstance(inflacion, pd.Series), "Falta la Series inflacion (Ej. 1)"
print("Checkpoint ✓ — todos los objetos del notebook están en su sitio")

---

## 🔁 Cómo fluye el análisis con Pandas

```
   Origen de datos              CSV / Excel / JSON / SQL / API
        │
        ▼
   Carga                        pd.read_csv() / pd.read_excel() / ...
        │
        ▼
   Auditoría rápida             .shape · .info() · .describe() · .isnull().sum()
        │
        ▼
   Selección                    df["col"] · df[[...]] · .loc · .iloc
        │
        ▼
   Filtrado                     df[df["col"] > X] · .query()
        │
        ▼
   Derivación                   df["nueva"] = df["a"] / df["b"]
        │
        ▼
   (Mañana) Agregación          .groupby() · .pivot_table() · .merge()
        │
        ▼
   Respuestas a preguntas de negocio
```

## 📚 Conceptos clave de hoy

| Concepto | Sintaxis | Para qué sirve |
|---|---|---|
| Series | `pd.Series([...], index=[...])` | Vector etiquetado (1D) |
| DataFrame | `pd.DataFrame({...})` | Tabla (2D) con tipos heterogéneos |
| Carga | `pd.read_csv("x.csv")` | Importar datos desde archivo |
| Forma | `.shape`, `.size` | Dimensiones del DataFrame |
| Tipos | `.dtypes`, `.info()` | Tipo y memoria por columna |
| Resumen | `.describe()` | Estadísticas descriptivas |
| Vistazo | `.head()`, `.tail()`, `.sample()` | Primeras/últimas/aleatorias filas |
| Selección col | `df["col"]`, `df[["a","b"]]` | Una o varias columnas |
| Selección etiqueta | `.loc[fila, col]` | Por nombre de índice/columna |
| Selección posición | `.iloc[i, j]` | Por índice numérico |
| Filtro booleano | `df[df["x"] > 0]` | Filas que cumplen una condición |
| Query string | `df.query("x > 0 and y < 5")` | Filtros legibles con strings |
| Crear columna | `df["nueva"] = ...` | Vectorizado, sin loops |
| Eliminar columna | `df.drop(columns=[...])` | Remover columnas |
| Cambiar índice | `.set_index()`, `.reset_index()` | Promover/demover columna a índice |
| Tabla cruzada | `pd.crosstab(filas, cols, values=, aggfunc=)` | Cruce rápido de dos variables |

## ⏭️ Para mañana — Pandas II

Mañana subimos un escalón:

- `groupby()` y agregaciones múltiples
- Limpieza de nulos (`fillna`, `dropna`) y duplicados
- Conversiones de tipo y columnas derivadas
- `merge()` y `concat()` para combinar datasets

**Para llegar caliente:**

- Vuelve a leer en `lecturas/tecnico-1-pandas-desde-cero-tutorial-completo.md` la sección **Selección y Indexación** si `loc`/`iloc` te confundieron.
- Practica con otros datasets: `sns.load_dataset("tips")`, `sns.load_dataset("flights")`, `sns.load_dataset("penguins")`.

## 📖 Referencias

- [Pandas — Getting Started](https://pandas.pydata.org/docs/getting_started/index.html) (oficial)
- [Pandas — Indexing & selecting](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) (recorrido oficial corto)
- Lecturas locales: `lecturas/tecnico-1-pandas-desde-cero-tutorial-completo.md` y `lecturas/tecnico-2-lectura-y-escritura-de-archivos-csv-con-pandas.md`

## ✍️ Reflexión

En tus propias palabras (máximo 3 oraciones), explícale a un compañero **por qué Pandas es mejor que listas de Python para datos tabulares**. Pista: piensa en velocidad, etiquetas y manejo de nulos.

> **Recordatorio** · Notebook ejecutado de principio a fin, subido al repo del equipo antes de la próxima sesión.

---

## 👥 Para tu equipo

Con el dataset de su caso de estudio:

1. **Carga** los datos con `pd.read_csv()` (o el lector apropiado).
2. **Auditoría rápida**:
   - `.shape`, `.info()`, `.describe()`
   - Porcentaje de nulos por columna
   - 5 filas aleatorias con `.sample(5)`
3. **Selección y filtros**:
   - Aíslen 2–3 subconjuntos con filtros booleanos relevantes para su pregunta de negocio.
   - Documenten cada filtro con un comentario que explique *por qué* es interesante.
4. **Tres preguntas de negocio**:
   - Identifiquen tres preguntas que el dataset puede responder y dejen escritas las **dos primeras consultas** que harían con Pandas para responderlas.

**Entregable:** `proyecto_exploracion_basica.ipynb` con la auditoría, los filtros y las preguntas.

---

*Seminario EDA · Día 6 · Martes 28 de abril de 2026*